# Module 03 — CrewAI Teams

> **Level:** Advanced | **Time:** ~90 min  
> **SDKs Used:** `crewai` (simulated patterns), `pydantic`, `dataclasses`

| Section | Topic |
|---------|-------|
| **Part 1** | Agents & Tasks — separating "Who" from "What" with typed output schemas |
| **Part 2** | Sequential vs Hierarchical — when to use each Process |
| **Part 3** | CrewAI Flows — event-driven state machines routing between Crews |
| **Part 4** | Production Checklist — anti-patterns and guardrails |

**Key thesis:** CrewAI agents should not "chat" — they produce typed artifacts.  
Tasks are the unit of work, not messages.


---
# Part 1: Agents & Tasks — Typed Artifacts over Chat

In CrewAI, `Agent` defines **who** can do the work (role, goal, tools), and `Task` defines **what** the deliverable is (description, `expected_output` schema). The key rule: tasks must produce **typed artifacts**, not conversational strings.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Optional
from pydantic import BaseModel
import json, time

# ─── Typed task output schemas ────────────────────────────────────────────────
class TelemetryArtifact(BaseModel):
    service: str
    error_rate: float
    cpu_p99: int
    top_errors: list[str]
    evidence_id: str

class DeploymentArtifact(BaseModel):
    service: str
    version: str
    deployed_at: str
    deployed_by: str
    changelog_url: str
    evidence_id: str

class IncidentBriefArtifact(BaseModel):
    hypothesis: str
    confidence: str        # "LOW" | "MEDIUM" | "HIGH"
    evidence_ids: list[str]
    affected_tenants: list[str]
    proposed_action: str
    requires_approval: bool

# ─── Simulated Agent ──────────────────────────────────────────────────────────
@dataclass
class Agent:
    role: str
    goal: str
    backstory: str
    tools: list[str] = field(default_factory=list)

    def __str__(self):
        return f"Agent(role={self.role!r}, tools={self.tools})"

# ─── Simulated Task ───────────────────────────────────────────────────────────
@dataclass
class Task:
    description: str
    expected_output: type    # The pydantic model we expect back
    agent: Agent
    context: list["Task"] = field(default_factory=list)  # depends on
    _result: Optional[Any] = field(default=None, init=False, repr=False)

    def execute(self) -> Any:
        """Run the task, optionally reading context from prior tasks."""
        context_data = [t._result for t in self.context if t._result]
        return self._simulate(context_data)

    def _simulate(self, ctx: list) -> Any:
        time.sleep(0.05)
        if self.expected_output == TelemetryArtifact:
            self._result = TelemetryArtifact(
                service="checkout-ui",
                error_rate=0.31,
                cpu_p99=94,
                top_errors=["3DS callback timeout", "VAT redirect 500"],
                evidence_id="EV-TELEM-001",
            )
        elif self.expected_output == DeploymentArtifact:
            self._result = DeploymentArtifact(
                service="checkout-ui",
                version="v2.1",
                deployed_at="2024-01-15T08:49:00Z",
                deployed_by="ci-bot@northstar.com",
                changelog_url="https://github.com/northstar/checkout-ui/releases/v2.1",
                evidence_id="EV-DEPLOY-001",
            )
        elif self.expected_output == IncidentBriefArtifact:
            ev_ids = [d["evidence_id"] for d in ctx if "evidence_id" in d]
            self._result = IncidentBriefArtifact(
                hypothesis="checkout-ui v2.1 introduced a broken 3DS VAT redirect for EU enterprise accounts.",
                confidence="HIGH",
                evidence_ids=ev_ids or ["EV-TELEM-001", "EV-DEPLOY-001"],
                affected_tenants=["northstar-eu-001", "globex-002", "acme-003"],
                proposed_action="Revert to v2.0 via feature-flag disable. Execute only after approval.",
                requires_approval=True,
            )
        return self._result

# ─── Demonstrate typed artifact handoff ──────────────────────────────────────
print("📦  CrewAI Typed Artifact Demo")
print("=" * 60)

obs_agent  = Agent("Observability Specialist", "Retrieve raw telemetry", "Expert in Datadog/Sentry", ["query_metrics", "search_logs"])
dep_agent  = Agent("Release Engineer",         "Retrieve deployment history", "Expert in GitHub Actions CI/CD", ["get_deployment"])
anl_agent  = Agent("Incident Analyst",         "Synthesise evidence into an incident brief", "Senior SRE, 10y experience", [])

task_telem  = Task("Fetch error rate and top errors for checkout-ui in last 30 min", TelemetryArtifact,  obs_agent)
task_deploy = Task("Fetch latest checkout-ui deployment metadata",                   DeploymentArtifact, dep_agent)
task_brief  = Task("Synthesise telemetry+deployment into incident brief",            IncidentBriefArtifact, anl_agent,
                   context=[task_telem, task_deploy])

for t in [task_telem, task_deploy, task_brief]:
    result = t.execute()
    print(f"\n  [{t.agent.role}] Task completed:")
    print(f"  Expected output : {t.expected_output.__name__}")
    print(f"  Artifact JSON   :\n{result.model_dump_json(indent=4)[:300]}...")

print("\n✅  All tasks completed with typed artifacts. No unstructured text passed between agents.")


📦  CrewAI Typed Artifact Demo

  [Observability Specialist] Task completed:
  Expected output : TelemetryArtifact
  Artifact JSON   :
{
    "service": "checkout-ui",
    "error_rate": 0.31,
    "cpu_p99": 94,
    "top_errors": [
        "3DS callback timeout",
        "VAT redirect 500"
    ],
    "evidence_id": "EV-TELEM-001"
}...

  [Release Engineer] Task completed:
  Expected output : DeploymentArtifact
  Artifact JSON   :
{
    "service": "checkout-ui",
    "version": "v2.1",
    "deployed_at": "2024-01-15T08:49:00Z",
    "deployed_by": "ci-bot@northstar.com",
    "changelog_url": "https://github.com/northstar/checkout-ui/releases/v2.1",
    "evidence_id": "EV-DEPLOY-001"
}...

  [Incident Analyst] Task completed:
  Expected output : IncidentBriefArtifact
  Artifact JSON   :
{
    "hypothesis": "checkout-ui v2.1 introduced a broken 3DS VAT redirect for EU enterprise accounts.",
    "confidence": "HIGH",
    "evidence_ids": [
        "EV-TELEM-001",
        "EV-DEPLOY-001"
    ],
 

---
# Part 2: Sequential vs Hierarchical Process

The `Process` determines execution order. `sequential` is predictable and cheap. `hierarchical` adds a Manager agent that can re-delegate — powerful but expensive.

In [2]:
import time, random
from dataclasses import dataclass, field
from typing import Optional, Callable

@dataclass
class TaskResult:
    task_name: str
    agent: str
    success: bool
    output: str
    attempts: int = 1

@dataclass
class SequentialCrew:
    """Tasks run in definition order. Each output passed to next as context."""
    tasks: list[tuple[str, str, bool]]  # (task_name, agent, can_fail)

    def kickoff(self, goal: str) -> list[TaskResult]:
        print(f"  🚀 Sequential Crew kickoff: {goal}")
        results = []
        for task_name, agent, can_fail in self.tasks:
            print(f"  [{agent}] → {task_name}...", end=" ")
            time.sleep(0.05)
            if can_fail and random.random() < 0.3:
                print("FAILED ❌")
                print(f"  ⛔  Pipeline halted at '{task_name}' — no recovery in sequential mode.")
                return results
            print("✅")
            results.append(TaskResult(task_name, agent, True, f"{agent} output for: {task_name}"))
        return results

@dataclass
class HierarchicalCrew:
    """A Manager agent dynamically delegates and can retry failed workers."""
    workers: dict[str, Callable]  # worker_name → work fn
    manager_model: str = "gpt-4o"

    def kickoff(self, goal: str) -> list[TaskResult]:
        print(f"  🚀 Hierarchical Crew kickoff: {goal}")
        print(f"  [Manager({self.manager_model})] Decomposing goal...")
        results = []
        sub_tasks = list(self.workers.items())

        for task_name, work_fn in sub_tasks:
            for attempt in range(1, 4):
                print(f"  [Manager] → Delegating '{task_name}' (attempt {attempt})...", end=" ")
                time.sleep(0.05)
                success = random.random() < (0.5 if attempt == 1 else 0.85)
                if success:
                    print("✅")
                    results.append(TaskResult(task_name, "worker", True, work_fn(), attempt))
                    break
                else:
                    print(f"FAILED — Manager retrying with revised delegation...")
            else:
                print(f"  [Manager] ⚠️  All attempts exhausted. Escalating '{task_name}'.")
                results.append(TaskResult(task_name, "worker", False, "ESCALATED", 3))
        return results

# ─── Run both approaches ──────────────────────────────────────────────────────
random.seed(7)
print("📋  Sequential Process (Process.sequential)")
print("=" * 60)
seq_crew = SequentialCrew(tasks=[
    ("fetch_metrics",    "ObservabilityAgent", False),
    ("fetch_deployment", "DeploymentAgent",    False),
    ("write_brief",      "AnalystAgent",       False),
])
seq_results = seq_crew.kickoff("Incident analysis for INC-2024-001")
print(f"\n  Completed: {len(seq_results)}/3 tasks  |  Mode: Sequential")

random.seed(8)
print("\n📋  Hierarchical Process (Process.hierarchical)")
print("=" * 60)
hier_crew = HierarchicalCrew(workers={
    "autonomous_research": lambda: "Research paper found via DuckDuckGo",
    "data_synthesis":      lambda: "Synthesised 12 data sources",
})
hier_results = hier_crew.kickoff("Autonomous market research on EU checkout UX")
print(f"\n  Completed: {sum(r.success for r in hier_results)}/{len(hier_results)} tasks  |  Mode: Hierarchical")

# Cost comparison
seq_cost  = sum(1 for r in seq_results) * 1.0    # 1x token per task
hier_cost = sum(r.attempts for r in hier_results) * 1.5  # Manager overhead
print(f"\n📊  Relative cost — Sequential: {seq_cost:.1f}x  Hierarchical: {hier_cost:.1f}x")


📋  Sequential Process (Process.sequential)
  🚀 Sequential Crew kickoff: Incident analysis for INC-2024-001
  [ObservabilityAgent] → fetch_metrics... 

✅
  [DeploymentAgent] → fetch_deployment... ✅
  [AnalystAgent] → write_brief... ✅

  Completed: 3/3 tasks  |  Mode: Sequential

📋  Hierarchical Process (Process.hierarchical)
  🚀 Hierarchical Crew kickoff: Autonomous market research on EU checkout UX
  [Manager(gpt-4o)] Decomposing goal...
  [Manager] → Delegating 'autonomous_research' (attempt 1)... 

✅
  [Manager] → Delegating 'data_synthesis' (attempt 1)... 

FAILED — Manager retrying with revised delegation...
  [Manager] → Delegating 'data_synthesis' (attempt 2)... ✅

  Completed: 2/2 tasks  |  Mode: Hierarchical

📊  Relative cost — Sequential: 3.0x  Hierarchical: 4.5x


---
# Part 3: CrewAI Flows — Event-Driven Routing

Flows wrap multiple Crews in a deterministic Python state machine. The `@start()` and `@listen()` decorators define the routing topology — no LLM decides which Crew runs.

In [3]:
from dataclasses import dataclass, field
from typing import Callable, Optional

# ─── Flow state (like CrewAI FlowState) ──────────────────────────────────────
@dataclass
class FlowState:
    intent: str = ""
    crew_output: Optional[str] = None
    routed_to: str = ""

# ─── Specialist Crews ─────────────────────────────────────────────────────────
class IncidentCrew:
    def kickoff(self) -> str:
        return "IncidentCrew: Root cause identified. Proposed revert to checkout-ui v2.0."

class ComplianceCrew:
    def kickoff(self) -> str:
        return "ComplianceCrew: GDPR data access request processed. Deletion confirmed."

class PerformanceCrew:
    def kickoff(self) -> str:
        return "PerformanceCrew: Query optimisation plan generated. Est. 40% latency reduction."

# ─── Flow implementation (mirrors CrewAI Flow decorators) ────────────────────
class TechSupportFlow:
    """
    Simulates a CrewAI Flow with @start and @listen routing.
    
    @start()         → classify_intent()
    @listen("incident")   → run_incident_crew()
    @listen("compliance") → run_compliance_crew()
    @listen("performance")→ run_performance_crew()
    """
    def __init__(self):
        self.state = FlowState()
        self._routes: dict[str, Callable] = {
            "incident":    self._run_incident,
            "compliance":  self._run_compliance,
            "performance": self._run_performance,
        }

    def classify_intent(self, user_input: str) -> str:
        """@start() — deterministic classification, never an LLM."""
        lower = user_input.lower()
        if any(k in lower for k in ["error", "down", "outage", "conversion", "failing"]):
            return "incident"
        elif any(k in lower for k in ["gdpr", "data", "delete", "compliance", "privacy"]):
            return "compliance"
        elif any(k in lower for k in ["slow", "latency", "performance", "query", "optimise"]):
            return "performance"
        return "incident"  # default fallback

    def _run_incident(self):    return IncidentCrew().kickoff()
    def _run_compliance(self):  return ComplianceCrew().kickoff()
    def _run_performance(self): return PerformanceCrew().kickoff()

    def kickoff(self, user_input: str) -> FlowState:
        print(f"  [Flow] Input: '{user_input[:60]}'")
        intent = self.classify_intent(user_input)
        print(f"  [Flow] @start() → classify_intent() → '{intent}'")
        print(f"  [Flow] @listen('{intent}') → routing to specialist Crew...")
        
        crew_fn = self._routes.get(intent, self._run_incident)
        output = crew_fn()
        
        self.state.intent = intent
        self.state.crew_output = output
        self.state.routed_to = intent
        
        print(f"  [Crew] Output: {output}")
        return self.state

# ─── Demo: route 3 different queries ─────────────────────────────────────────
flow = TechSupportFlow()
queries = [
    "EU checkout conversion is failing for enterprise accounts",
    "Customer requests GDPR data deletion for tenant northstar-eu-001",
    "Database queries are running slow, 4s p99 latency",
]

print("🔀  CrewAI Flow Demo — Event-Driven Routing")
print("=" * 60)

for q in queries:
    print(f"\nQuery: '{q[:55]}...'")
    result = flow.kickoff(q)
    print(f"  → Routed to: {result.routed_to.upper()} Crew")
    flow.state = FlowState()   # reset state for next query


🔀  CrewAI Flow Demo — Event-Driven Routing

Query: 'EU checkout conversion is failing for enterprise accoun...'
  [Flow] Input: 'EU checkout conversion is failing for enterprise accounts'
  [Flow] @start() → classify_intent() → 'incident'
  [Flow] @listen('incident') → routing to specialist Crew...
  [Crew] Output: IncidentCrew: Root cause identified. Proposed revert to checkout-ui v2.0.
  → Routed to: INCIDENT Crew

Query: 'Customer requests GDPR data deletion for tenant northst...'
  [Flow] Input: 'Customer requests GDPR data deletion for tenant northstar-eu'
  [Flow] @start() → classify_intent() → 'compliance'
  [Flow] @listen('compliance') → routing to specialist Crew...
  [Crew] Output: ComplianceCrew: GDPR data access request processed. Deletion confirmed.
  → Routed to: COMPLIANCE Crew

Query: 'Database queries are running slow, 4s p99 latency...'
  [Flow] Input: 'Database queries are running slow, 4s p99 latency'
  [Flow] @start() → classify_intent() → 'compliance'
  [Flow] @li

---
# Summary: CrewAI Production Checklist

| ✅ Do | ❌ Don't |
|------|---------|
| Define `expected_output` as a Pydantic schema | Pass raw conversational strings between tasks |
| Use `context=[task_a]` to declare explicit dependencies | Broadcast full chat history to every agent |
| Use `Process.sequential` for known linear workflows | Default to hierarchical — it's expensive |
| Reserve `Process.hierarchical` for ambiguous, retry-heavy work | Use hierarchical for ETL/reporting pipelines |
| Wrap multiple Crews in a `Flow` for intent routing | Put all logic in one giant monolithic Crew |
| Keep policy (auth, approval, RBAC) **outside** the Crew | Let agents decide their own authorization |
